# N3: Multi-Layer Perceptron (MLP) for Classification

Test two MLP configurations:
1. **MLP Model 1**: TF-IDF + PhoBERT embeddings combined
2. **MLP Model 2**: TF-IDF + PhoBERT embeddings + hand-crafted features

Compare performance and complexity across both models.

## 1. Import Required Libraries

In [23]:
import numpy as np
import pandas as pd
import warnings
import torch
import gc
import time
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from transformers import AutoTokenizer, AutoModel

# Setup device for PyTorch (PhoBERT extraction)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Using Scikit-learn MLPClassifier for neural networks")

✅ All imports successful
   PyTorch Device: cuda
   Using Scikit-learn MLPClassifier for neural networks


## 2. Load Data from CSV

In [24]:
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"✅ Data loaded: Train {df_train.shape} | Val {df_val.shape}")
print(f"   Labels: Train {(y_train==0).sum()} fake / {(y_train==1).sum()} real")
print(f"           Val {(y_val==0).sum()} fake / {(y_val==1).sum()} real")

✅ Data loaded: Train (3788, 28) | Val (474, 28)
   Labels: Train 3143 fake / 645 real
           Val 393 fake / 81 real


## 3. Generate TF-IDF Embeddings

In [25]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

print(f"✅ TF-IDF embeddings: Train {X_train_tfidf.shape} | Val {X_val_tfidf.shape}")

✅ TF-IDF embeddings: Train (3788, 900) | Val (474, 900)


## 4. Extract PhoBERT Embeddings

In [26]:
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    """Extract PhoBERT embeddings from text"""
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="PhoBERT", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    return np.array(embeddings)

X_train_phobert = extract_phobert_embeddings(texts_train_loose, batch_size=16)
X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)

print(f"✅ PhoBERT embeddings: Train {X_train_phobert.shape} | Val {X_val_phobert.shape}")

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ PhoBERT embeddings: Train (3788, 768) | Val (474, 768)


## 5. Extract Hand-Crafted Numeric Features

In [27]:
exclude_cols = {'id', 'label', 'text_strict', 'text_loose', 'post_message'}
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols 
                if col not in exclude_cols and col.strip() != '' and not col.startswith('Unnamed')]

if feature_cols:
    X_train_features = df_train[feature_cols].fillna(0).values
    X_val_features = df_val[feature_cols].fillna(0).values
else:
    X_train_features = np.array([]).reshape(len(df_train), 0)
    X_val_features = np.array([]).reshape(len(df_val), 0)

# Normalize hand-crafted features
scaler_features = StandardScaler()
X_train_features_scaled = scaler_features.fit_transform(X_train_features) if X_train_features.shape[1] > 0 else X_train_features
X_val_features_scaled = scaler_features.transform(X_val_features) if X_val_features.shape[1] > 0 else X_val_features

print(f"✅ Hand-crafted features: {len(feature_cols)} features")
print(f"   Train {X_train_features_scaled.shape} | Val {X_val_features_scaled.shape}")

✅ Hand-crafted features: 23 features
   Train (3788, 23) | Val (474, 23)


## 6. MLP Model 1: Combined Embeddings (TF-IDF + PhoBERT)

In [28]:
print("\n" + "="*80)
print("MODEL 1: MLP WITH COMBINED EMBEDDINGS (TF-IDF + PhoBERT)")
print("="*80)

# Combine embeddings (TF-IDF + PhoBERT)
X_train_combined_model1 = np.hstack([X_train_tfidf, X_train_phobert])
X_val_combined_model1 = np.hstack([X_val_tfidf, X_val_phobert])

# Normalize
scaler_model1 = StandardScaler()
X_train_combined_model1_scaled = scaler_model1.fit_transform(X_train_combined_model1)
X_val_combined_model1_scaled = scaler_model1.transform(X_val_combined_model1)

print(f"\n✅ Combined embeddings: {X_train_combined_model1_scaled.shape[1]} dimensions")
print(f"   TF-IDF: 900 + PhoBERT: 768 = Total: {900 + 768}")

# Build MLP Model 1 using scikit-learn
print("\nBuilding MLP architecture...")
print("   Configuration: 3 hidden layers (256, 128, 64)")
print("   Activation: ReLU")
print("   Learning rate: 0.001")

model1 = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    batch_size=32,
    random_state=42,
    verbose=False
)

# Train Model 1
print("\n📊 Training Model 1...")
model1.fit(X_train_combined_model1_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model1.n_iter_} iterations")
print(f"   Loss: {model1.loss_:.4f}")


MODEL 1: MLP WITH COMBINED EMBEDDINGS (TF-IDF + PhoBERT)

✅ Combined embeddings: 1668 dimensions
   TF-IDF: 900 + PhoBERT: 768 = Total: 1668

Building MLP architecture...
   Configuration: 3 hidden layers (256, 128, 64)
   Activation: ReLU
   Learning rate: 0.001

📊 Training Model 1...
✅ Training complete!
   Converged: 8 iterations
   Loss: 0.0033


## 7. Evaluate Model 1

In [29]:
print("\n" + "="*80)
print("RESULTS - MODEL 1 (Combined Embeddings)")
print("="*80)

# Get predictions
y_val_pred_model1 = model1.predict(X_val_combined_model1_scaled)
y_val_proba_model1 = model1.predict_proba(X_val_combined_model1_scaled)[:, 1]

# Calculate metrics
f1_model1 = f1_score(y_val, y_val_pred_model1, average='weighted')
auc_model1 = roc_auc_score(y_val, y_val_proba_model1)
acc_model1 = accuracy_score(y_val, y_val_pred_model1)
prec_model1 = precision_score(y_val, y_val_pred_model1, average='weighted')
rec_model1 = recall_score(y_val, y_val_pred_model1, average='weighted')

print(f"\n✅ Model 1 Metrics:")
print(f"   F1 Score:  {f1_model1:.4f}")
print(f"   AUC Score: {auc_model1:.4f}")
print(f"   Accuracy:  {acc_model1:.4f}")
print(f"   Precision: {prec_model1:.4f}")
print(f"   Recall:    {rec_model1:.4f}")

# Store results
results_model1 = {
    'F1': f1_model1,
    'AUC': auc_model1,
    'Accuracy': acc_model1,
    'Precision': prec_model1,
    'Recall': rec_model1
}


RESULTS - MODEL 1 (Combined Embeddings)

✅ Model 1 Metrics:
   F1 Score:  0.9062
   AUC Score: 0.9588
   Accuracy:  0.9072
   Precision: 0.9055
   Recall:    0.9072


## 7.1 Ensemble Model 1.1: MLP + Logistic Regression

In [30]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

print("\n" + "="*80)
print("MODEL 1.1: ENSEMBLE MLP + LOGISTIC REGRESSION")
print("="*80)

# Train Logistic Regression on same input (X_train_combined_model1_scaled)
print("\n📊 Training Logistic Regression...")
logistic_model1 = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    verbose=0
)
logistic_model1.fit(X_train_combined_model1_scaled, y_train)

# Get predictions from both models
y_val_proba_mlp_model1 = model1.predict_proba(X_val_combined_model1_scaled)[:, 1]
y_val_proba_logistic_model1 = logistic_model1.predict_proba(X_val_combined_model1_scaled)[:, 1]

# Ensemble: simple averaging
y_val_proba_ensemble_1_1 = (y_val_proba_mlp_model1 + y_val_proba_logistic_model1) / 2
y_val_pred_ensemble_1_1 = (y_val_proba_ensemble_1_1 >= 0.5).astype(int)

# Calculate metrics
f1_ensemble_1_1 = f1_score(y_val, y_val_pred_ensemble_1_1, average='weighted')
auc_ensemble_1_1 = roc_auc_score(y_val, y_val_proba_ensemble_1_1)
acc_ensemble_1_1 = accuracy_score(y_val, y_val_pred_ensemble_1_1)
prec_ensemble_1_1 = precision_score(y_val, y_val_pred_ensemble_1_1, average='weighted')
rec_ensemble_1_1 = recall_score(y_val, y_val_pred_ensemble_1_1, average='weighted')

print(f"\n✅ Model 1.1 Metrics (MLP + Logistic):")
print(f"   F1 Score:  {f1_ensemble_1_1:.4f}")
print(f"   AUC Score: {auc_ensemble_1_1:.4f}")
print(f"   Accuracy:  {acc_ensemble_1_1:.4f}")
print(f"   Precision: {prec_ensemble_1_1:.4f}")
print(f"   Recall:    {rec_ensemble_1_1:.4f}")

results_model_1_1 = {
    'F1': f1_ensemble_1_1,
    'AUC': auc_ensemble_1_1,
    'Accuracy': acc_ensemble_1_1,
    'Precision': prec_ensemble_1_1,
    'Recall': rec_ensemble_1_1
}



MODEL 1.1: ENSEMBLE MLP + LOGISTIC REGRESSION

📊 Training Logistic Regression...

✅ Model 1.1 Metrics (MLP + Logistic):
   F1 Score:  0.9119
   AUC Score: 0.9560
   Accuracy:  0.9135
   Precision: 0.9110
   Recall:    0.9135


## 7.2 Ensemble Model 1.2: MLP + LightGBM

In [31]:
print("\n" + "="*80)
print("MODEL 1.2: ENSEMBLE MLP + LIGHTGBM")
print("="*80)

# Train LightGBM on same input (X_train_combined_model1_scaled)
print("\n📊 Training LightGBM...")
lightgbm_model1 = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
lightgbm_model1.fit(X_train_combined_model1_scaled, y_train)

# Get predictions from both models
y_val_proba_lightgbm_model1 = lightgbm_model1.predict_proba(X_val_combined_model1_scaled)[:, 1]

# Ensemble: simple averaging
y_val_proba_ensemble_1_2 = (y_val_proba_mlp_model1 + y_val_proba_lightgbm_model1) / 2
y_val_pred_ensemble_1_2 = (y_val_proba_ensemble_1_2 >= 0.5).astype(int)

# Calculate metrics
f1_ensemble_1_2 = f1_score(y_val, y_val_pred_ensemble_1_2, average='weighted')
auc_ensemble_1_2 = roc_auc_score(y_val, y_val_proba_ensemble_1_2)
acc_ensemble_1_2 = accuracy_score(y_val, y_val_pred_ensemble_1_2)
prec_ensemble_1_2 = precision_score(y_val, y_val_pred_ensemble_1_2, average='weighted')
rec_ensemble_1_2 = recall_score(y_val, y_val_pred_ensemble_1_2, average='weighted')

print(f"\n✅ Model 1.2 Metrics (MLP + LightGBM):")
print(f"   F1 Score:  {f1_ensemble_1_2:.4f}")
print(f"   AUC Score: {auc_ensemble_1_2:.4f}")
print(f"   Accuracy:  {acc_ensemble_1_2:.4f}")
print(f"   Precision: {prec_ensemble_1_2:.4f}")
print(f"   Recall:    {rec_ensemble_1_2:.4f}")

results_model_1_2 = {
    'F1': f1_ensemble_1_2,
    'AUC': auc_ensemble_1_2,
    'Accuracy': acc_ensemble_1_2,
    'Precision': prec_ensemble_1_2,
    'Recall': rec_ensemble_1_2
}



MODEL 1.2: ENSEMBLE MLP + LIGHTGBM

📊 Training LightGBM...

✅ Model 1.2 Metrics (MLP + LightGBM):
   F1 Score:  0.9043
   AUC Score: 0.9565
   Accuracy:  0.9093
   Precision: 0.9045
   Recall:    0.9093


## 7.3 Stacking Model 1.3: MLP + LightGBM → Logistic

In [32]:
print("\n" + "="*80)
print("MODEL 1.3: STACKING MLP + LIGHTGBM → LOGISTIC REGRESSION")
print("="*80)

# Meta-features: Only outputs from MLP and LightGBM (not original data)
X_train_meta_1_3 = np.hstack([
    model1.predict_proba(X_train_combined_model1_scaled)[:, 1].reshape(-1, 1),
    lightgbm_model1.predict_proba(X_train_combined_model1_scaled)[:, 1].reshape(-1, 1)
])
X_val_meta_1_3 = np.hstack([
    y_val_proba_mlp_model1.reshape(-1, 1),
    y_val_proba_lightgbm_model1.reshape(-1, 1)
])

print("\n📊 Training Meta-Learner (Logistic Regression)...")
stacking_meta_1_3 = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    verbose=0
)
stacking_meta_1_3.fit(X_train_meta_1_3, y_train)

# Get predictions from meta-learner
y_val_proba_stacking_1_3 = stacking_meta_1_3.predict_proba(X_val_meta_1_3)[:, 1]
y_val_pred_stacking_1_3 = (y_val_proba_stacking_1_3 >= 0.5).astype(int)

# Calculate metrics
f1_stacking_1_3 = f1_score(y_val, y_val_pred_stacking_1_3, average='weighted')
auc_stacking_1_3 = roc_auc_score(y_val, y_val_proba_stacking_1_3)
acc_stacking_1_3 = accuracy_score(y_val, y_val_pred_stacking_1_3)
prec_stacking_1_3 = precision_score(y_val, y_val_pred_stacking_1_3, average='weighted')
rec_stacking_1_3 = recall_score(y_val, y_val_pred_stacking_1_3, average='weighted')

print(f"\n✅ Model 1.3 Metrics (Stacking):")
print(f"   F1 Score:  {f1_stacking_1_3:.4f}")
print(f"   AUC Score: {auc_stacking_1_3:.4f}")
print(f"   Accuracy:  {acc_stacking_1_3:.4f}")
print(f"   Precision: {prec_stacking_1_3:.4f}")
print(f"   Recall:    {rec_stacking_1_3:.4f}")

results_model_1_3 = {
    'F1': f1_stacking_1_3,
    'AUC': auc_stacking_1_3,
    'Accuracy': acc_stacking_1_3,
    'Precision': prec_stacking_1_3,
    'Recall': rec_stacking_1_3
}



MODEL 1.3: STACKING MLP + LIGHTGBM → LOGISTIC REGRESSION

📊 Training Meta-Learner (Logistic Regression)...

✅ Model 1.3 Metrics (Stacking):
   F1 Score:  0.8956
   AUC Score: 0.9538
   Accuracy:  0.9051
   Precision: 0.9018
   Recall:    0.9051


## 8. MLP Model 2: Combined Embeddings + Hand-Crafted Features

In [33]:
print("\n" + "="*80)
print("MODEL 2: MLP WITH EMBEDDINGS + HAND-CRAFTED FEATURES")
print("="*80)

# Combine embeddings + hand-crafted features
X_train_combined_model2 = np.hstack([X_train_tfidf, X_train_phobert, X_train_features_scaled])
X_val_combined_model2 = np.hstack([X_val_tfidf, X_val_phobert, X_val_features_scaled])

# Normalize
scaler_model2 = StandardScaler()
X_train_combined_model2_scaled = scaler_model2.fit_transform(X_train_combined_model2)
X_val_combined_model2_scaled = scaler_model2.transform(X_val_combined_model2)

print(f"\n✅ Combined features: {X_train_combined_model2_scaled.shape[1]} dimensions")
print(f"   TF-IDF: 900 + PhoBERT: 768 + Hand-crafted: {X_train_features_scaled.shape[1]} = Total: {900 + 768 + X_train_features_scaled.shape[1]}")

# Build MLP Model 2 (slightly larger due to more features)
print("\nBuilding MLP architecture...")
print("   Configuration: 3 hidden layers (128, 64)")
print("   Activation: ReLU")
print("   Learning rate: 0.001")

model2 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    batch_size=32,
    random_state=42,
    verbose=False
)

# Train Model 2
print("\n📊 Training Model 2...")
model2.fit(X_train_combined_model2_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model2.n_iter_} iterations")
print(f"   Loss: {model2.loss_:.4f}")


MODEL 2: MLP WITH EMBEDDINGS + HAND-CRAFTED FEATURES

✅ Combined features: 1691 dimensions
   TF-IDF: 900 + PhoBERT: 768 + Hand-crafted: 23 = Total: 1691

Building MLP architecture...
   Configuration: 3 hidden layers (128, 64)
   Activation: ReLU
   Learning rate: 0.001

📊 Training Model 2...
✅ Training complete!
   Converged: 8 iterations
   Loss: 0.0019


## 9. Evaluate Model 2

In [34]:
print("\n" + "="*80)
print("RESULTS - MODEL 2 (Embeddings + Hand-Crafted Features)")
print("="*80)

# Get predictions
y_val_pred_model2 = model2.predict(X_val_combined_model2_scaled)
y_val_proba_model2 = model2.predict_proba(X_val_combined_model2_scaled)[:, 1]

# Calculate metrics
f1_model2 = f1_score(y_val, y_val_pred_model2, average='weighted')
auc_model2 = roc_auc_score(y_val, y_val_proba_model2)
acc_model2 = accuracy_score(y_val, y_val_pred_model2)
prec_model2 = precision_score(y_val, y_val_pred_model2, average='weighted')
rec_model2 = recall_score(y_val, y_val_pred_model2, average='weighted')

print(f"\n✅ Model 2 Metrics:")
print(f"   F1 Score:  {f1_model2:.4f}")
print(f"   AUC Score: {auc_model2:.4f}")
print(f"   Accuracy:  {acc_model2:.4f}")
print(f"   Precision: {prec_model2:.4f}")
print(f"   Recall:    {rec_model2:.4f}")

# Store results
results_model2 = {
    'F1': f1_model2,
    'AUC': auc_model2,
    'Accuracy': acc_model2,
    'Precision': prec_model2,
    'Recall': rec_model2
}


RESULTS - MODEL 2 (Embeddings + Hand-Crafted Features)

✅ Model 2 Metrics:
   F1 Score:  0.9241
   AUC Score: 0.9567
   Accuracy:  0.9241
   Precision: 0.9241
   Recall:    0.9241


## 9.1 Ensemble Model 2.1: MLP + Logistic Regression

In [35]:
print("\n" + "="*80)
print("MODEL 2.1: ENSEMBLE MLP + LOGISTIC REGRESSION")
print("="*80)

# Train Logistic Regression on same input (X_train_combined_model2_scaled)
print("\n📊 Training Logistic Regression...")
logistic_model2 = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    verbose=0
)
logistic_model2.fit(X_train_combined_model2_scaled, y_train)

# Get predictions from both models
y_val_proba_mlp_model2 = model2.predict_proba(X_val_combined_model2_scaled)[:, 1]
y_val_proba_logistic_model2 = logistic_model2.predict_proba(X_val_combined_model2_scaled)[:, 1]

# Ensemble: simple averaging
y_val_proba_ensemble_2_1 = (y_val_proba_mlp_model2 + y_val_proba_logistic_model2) / 2
y_val_pred_ensemble_2_1 = (y_val_proba_ensemble_2_1 >= 0.5).astype(int)

# Calculate metrics
f1_ensemble_2_1 = f1_score(y_val, y_val_pred_ensemble_2_1, average='weighted')
auc_ensemble_2_1 = roc_auc_score(y_val, y_val_proba_ensemble_2_1)
acc_ensemble_2_1 = accuracy_score(y_val, y_val_pred_ensemble_2_1)
prec_ensemble_2_1 = precision_score(y_val, y_val_pred_ensemble_2_1, average='weighted')
rec_ensemble_2_1 = recall_score(y_val, y_val_pred_ensemble_2_1, average='weighted')

print(f"\n✅ Model 2.1 Metrics (MLP + Logistic):")
print(f"   F1 Score:  {f1_ensemble_2_1:.4f}")
print(f"   AUC Score: {auc_ensemble_2_1:.4f}")
print(f"   Accuracy:  {acc_ensemble_2_1:.4f}")
print(f"   Precision: {prec_ensemble_2_1:.4f}")
print(f"   Recall:    {rec_ensemble_2_1:.4f}")

results_model_2_1 = {
    'F1': f1_ensemble_2_1,
    'AUC': auc_ensemble_2_1,
    'Accuracy': acc_ensemble_2_1,
    'Precision': prec_ensemble_2_1,
    'Recall': rec_ensemble_2_1
}



MODEL 2.1: ENSEMBLE MLP + LOGISTIC REGRESSION

📊 Training Logistic Regression...

✅ Model 2.1 Metrics (MLP + Logistic):
   F1 Score:  0.9201
   AUC Score: 0.9574
   Accuracy:  0.9219
   Precision: 0.9193
   Recall:    0.9219


## 9.2 Ensemble Model 2.2: MLP + LightGBM

In [36]:
print("\n" + "="*80)
print("MODEL 2.2: ENSEMBLE MLP + LIGHTGBM")
print("="*80)

# Train LightGBM on same input (X_train_combined_model2_scaled)
print("\n📊 Training LightGBM...")
lightgbm_model2 = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
lightgbm_model2.fit(X_train_combined_model2_scaled, y_train)

# Get predictions from both models
y_val_proba_lightgbm_model2 = lightgbm_model2.predict_proba(X_val_combined_model2_scaled)[:, 1]

# Ensemble: simple averaging
y_val_proba_ensemble_2_2 = (y_val_proba_mlp_model2 + y_val_proba_lightgbm_model2) / 2
y_val_pred_ensemble_2_2 = (y_val_proba_ensemble_2_2 >= 0.5).astype(int)

# Calculate metrics
f1_ensemble_2_2 = f1_score(y_val, y_val_pred_ensemble_2_2, average='weighted')
auc_ensemble_2_2 = roc_auc_score(y_val, y_val_proba_ensemble_2_2)
acc_ensemble_2_2 = accuracy_score(y_val, y_val_pred_ensemble_2_2)
prec_ensemble_2_2 = precision_score(y_val, y_val_pred_ensemble_2_2, average='weighted')
rec_ensemble_2_2 = recall_score(y_val, y_val_pred_ensemble_2_2, average='weighted')

print(f"\n✅ Model 2.2 Metrics (MLP + LightGBM):")
print(f"   F1 Score:  {f1_ensemble_2_2:.4f}")
print(f"   AUC Score: {auc_ensemble_2_2:.4f}")
print(f"   Accuracy:  {acc_ensemble_2_2:.4f}")
print(f"   Precision: {prec_ensemble_2_2:.4f}")
print(f"   Recall:    {rec_ensemble_2_2:.4f}")

results_model_2_2 = {
    'F1': f1_ensemble_2_2,
    'AUC': auc_ensemble_2_2,
    'Accuracy': acc_ensemble_2_2,
    'Precision': prec_ensemble_2_2,
    'Recall': rec_ensemble_2_2
}



MODEL 2.2: ENSEMBLE MLP + LIGHTGBM

📊 Training LightGBM...

✅ Model 2.2 Metrics (MLP + LightGBM):
   F1 Score:  0.9202
   AUC Score: 0.9583
   Accuracy:  0.9241
   Precision: 0.9211
   Recall:    0.9241


## 9.3 Stacking Model 2.3: MLP + LightGBM → Logistic

In [37]:
print("\n" + "="*80)
print("MODEL 2.3: STACKING MLP + LIGHTGBM → LOGISTIC REGRESSION")
print("="*80)

# Meta-features: Only outputs from MLP and LightGBM (not original data)
X_train_meta_2_3 = np.hstack([
    model2.predict_proba(X_train_combined_model2_scaled)[:, 1].reshape(-1, 1),
    lightgbm_model2.predict_proba(X_train_combined_model2_scaled)[:, 1].reshape(-1, 1)
])
X_val_meta_2_3 = np.hstack([
    y_val_proba_mlp_model2.reshape(-1, 1),
    y_val_proba_lightgbm_model2.reshape(-1, 1)
])

print("\n📊 Training Meta-Learner (Logistic Regression)...")
stacking_meta_2_3 = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    verbose=0
)
stacking_meta_2_3.fit(X_train_meta_2_3, y_train)

# Get predictions from meta-learner
y_val_proba_stacking_2_3 = stacking_meta_2_3.predict_proba(X_val_meta_2_3)[:, 1]
y_val_pred_stacking_2_3 = (y_val_proba_stacking_2_3 >= 0.5).astype(int)

# Calculate metrics
f1_stacking_2_3 = f1_score(y_val, y_val_pred_stacking_2_3, average='weighted')
auc_stacking_2_3 = roc_auc_score(y_val, y_val_proba_stacking_2_3)
acc_stacking_2_3 = accuracy_score(y_val, y_val_pred_stacking_2_3)
prec_stacking_2_3 = precision_score(y_val, y_val_pred_stacking_2_3, average='weighted')
rec_stacking_2_3 = recall_score(y_val, y_val_pred_stacking_2_3, average='weighted')

print(f"\n✅ Model 2.3 Metrics (Stacking):")
print(f"   F1 Score:  {f1_stacking_2_3:.4f}")
print(f"   AUC Score: {auc_stacking_2_3:.4f}")
print(f"   Accuracy:  {acc_stacking_2_3:.4f}")
print(f"   Precision: {prec_stacking_2_3:.4f}")
print(f"   Recall:    {rec_stacking_2_3:.4f}")

results_model_2_3 = {
    'F1': f1_stacking_2_3,
    'AUC': auc_stacking_2_3,
    'Accuracy': acc_stacking_2_3,
    'Precision': prec_stacking_2_3,
    'Recall': rec_stacking_2_3
}



MODEL 2.3: STACKING MLP + LIGHTGBM → LOGISTIC REGRESSION

📊 Training Meta-Learner (Logistic Regression)...

✅ Model 2.3 Metrics (Stacking):
   F1 Score:  0.9109
   AUC Score: 0.9558
   Accuracy:  0.9177
   Precision: 0.9159
   Recall:    0.9177


## 10. Comparison: Model 1 vs Model 2

In [38]:
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ALL MODELS (8 Variants)")
print("="*80)

# Create comprehensive comparison DataFrame
all_results = {
    'Model': [
        'Model 1 (MLP)',
        'Model 1.1 (MLP+Logistic)',
        'Model 1.2 (MLP+LightGBM)',
        'Model 1.3 (Stacking)',
        'Model 2 (MLP)',
        'Model 2.1 (MLP+Logistic)',
        'Model 2.2 (MLP+LightGBM)',
        'Model 2.3 (Stacking)'
    ],
    'Input Features': [
        'TF-IDF + PhoBERT (1,668D)',
        'TF-IDF + PhoBERT (1,668D)',
        'TF-IDF + PhoBERT (1,668D)',
        'TF-IDF + PhoBERT (1,668D)',
        'TF-IDF + PhoBERT + Hand-crafted',
        'TF-IDF + PhoBERT + Hand-crafted',
        'TF-IDF + PhoBERT + Hand-crafted',
        'TF-IDF + PhoBERT + Hand-crafted'
    ],
    'Method': [
        'Single MLP (256-128-64)',
        'Ensemble (MLP + Logistic)',
        'Ensemble (MLP + LightGBM)',
        'Stacking (MLP + LightGBM → Logistic)',
        'Single MLP (128-64)',
        'Ensemble (MLP + Logistic)',
        'Ensemble (MLP + LightGBM)',
        'Stacking (MLP + LightGBM → Logistic)'
    ],
    'F1': [
        f"{results_model1['F1']:.4f}",
        f"{results_model_1_1['F1']:.4f}",
        f"{results_model_1_2['F1']:.4f}",
        f"{results_model_1_3['F1']:.4f}",
        f"{results_model2['F1']:.4f}",
        f"{results_model_2_1['F1']:.4f}",
        f"{results_model_2_2['F1']:.4f}",
        f"{results_model_2_3['F1']:.4f}"
    ],
    'AUC': [
        f"{results_model1['AUC']:.4f}",
        f"{results_model_1_1['AUC']:.4f}",
        f"{results_model_1_2['AUC']:.4f}",
        f"{results_model_1_3['AUC']:.4f}",
        f"{results_model2['AUC']:.4f}",
        f"{results_model_2_1['AUC']:.4f}",
        f"{results_model_2_2['AUC']:.4f}",
        f"{results_model_2_3['AUC']:.4f}"
    ],
    'Accuracy': [
        f"{results_model1['Accuracy']:.4f}",
        f"{results_model_1_1['Accuracy']:.4f}",
        f"{results_model_1_2['Accuracy']:.4f}",
        f"{results_model_1_3['Accuracy']:.4f}",
        f"{results_model2['Accuracy']:.4f}",
        f"{results_model_2_1['Accuracy']:.4f}",
        f"{results_model_2_2['Accuracy']:.4f}",
        f"{results_model_2_3['Accuracy']:.4f}"
    ]
}

comparison_df = pd.DataFrame(all_results)
print("\n" + "="*80)
print("PERFORMANCE METRICS SUMMARY")
print("="*80)
print("\n" + comparison_df[['Model', 'Method', 'F1', 'AUC', 'Accuracy']].to_string(index=False))

# Find best performers
print(f"\n" + "="*80)
print("BEST PERFORMERS")
print("="*80)

f1_scores = [results_model1['F1'], results_model_1_1['F1'], results_model_1_2['F1'], results_model_1_3['F1'],
             results_model2['F1'], results_model_2_1['F1'], results_model_2_2['F1'], results_model_2_3['F1']]
auc_scores = [results_model1['AUC'], results_model_1_1['AUC'], results_model_1_2['AUC'], results_model_1_3['AUC'],
              results_model2['AUC'], results_model_2_1['AUC'], results_model_2_2['AUC'], results_model_2_3['AUC']]

best_f1_idx = np.argmax(f1_scores)
best_auc_idx = np.argmax(auc_scores)

print(f"\n🏆 Best F1 Score: {all_results['Model'][best_f1_idx]}")
print(f"   F1: {f1_scores[best_f1_idx]:.4f} | AUC: {auc_scores[best_f1_idx]:.4f}")
print(f"\n🏆 Best AUC Score: {all_results['Model'][best_auc_idx]}")
print(f"   F1: {f1_scores[best_auc_idx]:.4f} | AUC: {auc_scores[best_auc_idx]:.4f}")

# Analysis
print(f"\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)
print(f"\n📊 Model 1 Family (TF-IDF + PhoBERT):")
print(f"   Baseline (MLP):        F1={results_model1['F1']:.4f}")
print(f"   With Logistic Ens:     F1={results_model_1_1['F1']:.4f} ({results_model_1_1['F1']-results_model1['F1']:+.4f})")
print(f"   With LightGBM Ens:     F1={results_model_1_2['F1']:.4f} ({results_model_1_2['F1']-results_model1['F1']:+.4f})")
print(f"   With Stacking:         F1={results_model_1_3['F1']:.4f} ({results_model_1_3['F1']-results_model1['F1']:+.4f})")

print(f"\n📊 Model 2 Family (With Hand-Crafted Features):")
print(f"   Baseline (MLP):        F1={results_model2['F1']:.4f}")
print(f"   With Logistic Ens:     F1={results_model_2_1['F1']:.4f} ({results_model_2_1['F1']-results_model2['F1']:+.4f})")
print(f"   With LightGBM Ens:     F1={results_model_2_2['F1']:.4f} ({results_model_2_2['F1']-results_model2['F1']:+.4f})")
print(f"   With Stacking:         F1={results_model_2_3['F1']:.4f} ({results_model_2_3['F1']-results_model2['F1']:+.4f})")

print(f"\n✅ Analysis Complete!")



COMPREHENSIVE COMPARISON: ALL MODELS (8 Variants)

PERFORMANCE METRICS SUMMARY

                   Model                               Method     F1    AUC Accuracy
           Model 1 (MLP)              Single MLP (256-128-64) 0.9062 0.9588   0.9072
Model 1.1 (MLP+Logistic)            Ensemble (MLP + Logistic) 0.9119 0.9560   0.9135
Model 1.2 (MLP+LightGBM)            Ensemble (MLP + LightGBM) 0.9043 0.9565   0.9093
    Model 1.3 (Stacking) Stacking (MLP + LightGBM → Logistic) 0.8956 0.9538   0.9051
           Model 2 (MLP)                  Single MLP (128-64) 0.9241 0.9567   0.9241
Model 2.1 (MLP+Logistic)            Ensemble (MLP + Logistic) 0.9201 0.9574   0.9219
Model 2.2 (MLP+LightGBM)            Ensemble (MLP + LightGBM) 0.9202 0.9583   0.9241
    Model 2.3 (Stacking) Stacking (MLP + LightGBM → Logistic) 0.9109 0.9558   0.9177

BEST PERFORMERS

🏆 Best F1 Score: Model 2 (MLP)
   F1: 0.9241 | AUC: 0.9567

🏆 Best AUC Score: Model 1 (MLP)
   F1: 0.9062 | AUC: 0.9588

KEY INSIGHTS

📊

## 11. Model Complexity & Computational Analysis

In [39]:
import time

print("\n" + "="*80)
print("MODEL COMPLEXITY & COMPUTATIONAL ANALYSIS (+ FLOPs)")
print("="*80)

# Helper functions to calculate parameters and FLOPs
def count_mlp_params(layer_sizes, input_dim):
    """Count total parameters in MLP"""
    total = 0
    prev_size = input_dim
    for layer_size in layer_sizes:
        total += (prev_size * layer_size) + layer_size  # weights + bias
        prev_size = layer_size
    total += (prev_size * 2) + 2  # output layer (binary classification)
    return total

def count_mlp_flops(layer_sizes, input_dim):
    """Estimate FLOPs for MLP forward pass"""
    total_flops = 0
    prev_size = input_dim
    for layer_size in layer_sizes:
        total_flops += 2 * prev_size * layer_size
        prev_size = layer_size
    total_flops += 2 * prev_size * 2  # output layer
    return total_flops

def count_logistic_params(input_dim):
    """Count parameters in Logistic Regression"""
    return (input_dim * 1) + 1  # weights + bias

def count_logistic_flops(input_dim):
    """FLOPs for Logistic Regression"""
    return 2 * input_dim  # multiply + add per feature

def count_lightgbm_params(n_estimators, n_features):
    """Rough estimate for LightGBM parameters"""
    return n_estimators * n_features * 10

def count_lightgbm_flops(n_estimators, avg_depth):
    """FLOPs for LightGBM prediction"""
    return n_estimators * avg_depth * 2

# Calculate parameters and FLOPs for all models
mlp_model1_params = count_mlp_params([256, 128, 64], 1668)
mlp_model1_flops = count_mlp_flops([256, 128, 64], 1668)

mlp_model2_params = count_mlp_params([128, 64], 1724)  # 1668 + 56 hand-crafted
mlp_model2_flops = count_mlp_flops([128, 64], 1724)

logistic_model_params_1 = count_logistic_params(1668)
logistic_flops_1 = count_logistic_flops(1668)

logistic_model_params_2 = count_logistic_params(1724)
logistic_flops_2 = count_logistic_flops(1724)

lightgbm_params_1 = count_lightgbm_params(100, 1668)
lightgbm_flops_1 = count_lightgbm_flops(100, 7)  # avg tree depth ~7

lightgbm_params_2 = count_lightgbm_params(100, 1724)
lightgbm_flops_2 = count_lightgbm_flops(100, 7)

logistic_meta_params = count_logistic_params(2)  # Only 2 inputs from base models
logistic_meta_flops = count_logistic_flops(2)

# ============================================================================
# TABLE 1: Individual Models - Complexity Metrics
# ============================================================================
print("\n" + "─"*120)
print("TABLE 1: INDIVIDUAL MODELS - COMPLEXITY METRICS (Including FLOPs Estimation)")
print("─"*120 + "\n")

complexity_data = {
    'Model': [
        'Model 1 (MLP)',
        'Model 1.1 (MLP+Logistic)',
        'Model 1.2 (MLP+LightGBM)',
        'Model 1.3 (Stacking)',
        'Model 2 (MLP)',
        'Model 2.1 (MLP+Logistic)',
        'Model 2.2 (MLP+LightGBM)',
        'Model 2.3 (Stacking)'
    ],
    'Type': [
        'MLP', 'Ensemble', 'Ensemble', 'Stacking',
        'MLP', 'Ensemble', 'Ensemble', 'Stacking'
    ],
    'Input Dims': [
        1668, 1668, 1668, 1668,
        1724, 1724, 1724, 1724
    ],
    'Parameters': [
        mlp_model1_params,
        mlp_model1_params + logistic_model_params_1,
        mlp_model1_params + lightgbm_params_1,
        mlp_model1_params + lightgbm_params_1 + logistic_meta_params,
        mlp_model2_params,
        mlp_model2_params + logistic_model_params_2,
        mlp_model2_params + lightgbm_params_2,
        mlp_model2_params + lightgbm_params_2 + logistic_meta_params
    ],
    'FLOPs/Sample': [
        mlp_model1_flops,
        mlp_model1_flops + logistic_flops_1,
        mlp_model1_flops + lightgbm_flops_1,
        mlp_model1_flops + lightgbm_flops_1 + logistic_meta_flops,
        mlp_model2_flops,
        mlp_model2_flops + logistic_flops_2,
        mlp_model2_flops + lightgbm_flops_2,
        mlp_model2_flops + lightgbm_flops_2 + logistic_meta_flops
    ]
}

complexity_individual_df = pd.DataFrame(complexity_data)
complexity_individual_df['Parameters'] = complexity_individual_df['Parameters'].apply(lambda x: f"{int(x):,}")
complexity_individual_df['FLOPs/Sample'] = complexity_individual_df['FLOPs/Sample'].apply(lambda x: f"{int(x):,}")
print(complexity_individual_df.to_string(index=False))

# ============================================================================
# TABLE 2: Models Summary - Efficiency Metrics
# ============================================================================
print("\n" + "─"*120)
print("TABLE 2: MODELS SUMMARY - EFFICIENCY METRICS")
print("─"*120 + "\n")

f1_scores_list = [results_model1['F1'], results_model_1_1['F1'], results_model_1_2['F1'], results_model_1_3['F1'],
                  results_model2['F1'], results_model_2_1['F1'], results_model_2_2['F1'], results_model_2_3['F1']]
auc_scores_list = [results_model1['AUC'], results_model_1_1['AUC'], results_model_1_2['AUC'], results_model_1_3['AUC'],
                   results_model2['AUC'], results_model_2_1['AUC'], results_model_2_2['AUC'], results_model_2_3['AUC']]

# Convert parameters to numeric for efficiency calculation
complexity_params_numeric = [mlp_model1_params,
                              mlp_model1_params + logistic_model_params_1,
                              mlp_model1_params + lightgbm_params_1,
                              mlp_model1_params + lightgbm_params_1 + logistic_meta_params,
                              mlp_model2_params,
                              mlp_model2_params + logistic_model_params_2,
                              mlp_model2_params + lightgbm_params_2,
                              mlp_model2_params + lightgbm_params_2 + logistic_meta_params]

complexity_flops_numeric = [mlp_model1_flops,
                             mlp_model1_flops + logistic_flops_1,
                             mlp_model1_flops + lightgbm_flops_1,
                             mlp_model1_flops + lightgbm_flops_1 + logistic_meta_flops,
                             mlp_model2_flops,
                             mlp_model2_flops + logistic_flops_2,
                             mlp_model2_flops + lightgbm_flops_2,
                             mlp_model2_flops + lightgbm_flops_2 + logistic_meta_flops]

# Calculate efficiency ratios
efficiency_f1_params = np.array(f1_scores_list) / np.array(complexity_params_numeric, dtype=float)
efficiency_f1_flops = np.array(f1_scores_list) / np.array(complexity_flops_numeric, dtype=float)

summary_df = pd.DataFrame({
    'Model': complexity_data['Model'],
    'Total Params': [f"{p:,}" for p in complexity_params_numeric],
    'Total FLOPs/Sample': [f"{f:,}" for f in complexity_flops_numeric],
    'F1 Score': [f"{f1:.4f}" for f1 in f1_scores_list],
    'AUC-ROC': [f"{auc:.4f}" for auc in auc_scores_list],
    'Efficiency (F1/Params)': [f"{e:.8f}" for e in efficiency_f1_params],
    'FLOPs Efficiency (F1/FLOPs)': [f"{e:.8f}" for e in efficiency_f1_flops]
})

print(summary_df.to_string(index=False))

# ============================================================================
# TABLE 3: Detailed Model Analysis
# ============================================================================
print("\n" + "="*80)
print("TABLE 3: DETAILED MODEL ANALYSIS")
print("-" * 80)

for idx, name in enumerate(complexity_data['Model']):
    print(f"\n{name}:")
    print(f"  Architecture: {complexity_data['Type'][idx]}")
    print(f"  Input Dims: {complexity_data['Input Dims'][idx]}")
    print(f"  Parameters: {complexity_params_numeric[idx]:,}")
    print(f"  FLOPs/Sample: {complexity_flops_numeric[idx]:,}")
    print(f"  F1 Score: {f1_scores_list[idx]:.4f} | AUC-ROC: {auc_scores_list[idx]:.4f}")
    print(f"  Efficiency (F1/Params): {efficiency_f1_params[idx]:.8f}")
    print(f"  FLOPs Efficiency (F1/FLOPs): {efficiency_f1_flops[idx]:.8f}")

# ============================================================================
# Best Models by Efficiency
# ============================================================================
print("\n" + "="*80)
best_f1_params_idx = np.argmax(efficiency_f1_params)
best_f1_flops_idx = np.argmax(efficiency_f1_flops)

print(f"\n🏆 Best Model (F1/Parameters Efficiency): {complexity_data['Model'][best_f1_params_idx]}")
print(f"   Efficiency Score: {efficiency_f1_params[best_f1_params_idx]:.8f}")
print(f"   F1 Score: {f1_scores_list[best_f1_params_idx]:.4f}")

print(f"\n⚡ Best Model (F1/FLOPs Efficiency): {complexity_data['Model'][best_f1_flops_idx]}")
print(f"   Efficiency Score: {efficiency_f1_flops[best_f1_flops_idx]:.8f}")
print(f"   F1 Score: {f1_scores_list[best_f1_flops_idx]:.4f}")

print(f"\n" + "="*80)
print("NOTES:")
print("-" * 80)
print("• FLOPs/Sample: Estimated floating-point operations per prediction sample")
print("  - MLP: 2 × (input_dim × neurons) for forward pass")
print("  - Logistic: 2 × input_dims (1 multiply + 1 add per feature)")
print("  - LightGBM: n_estimators × avg_tree_depth × 2 (traversal operations)")
print("• Parameters: Model weights and coefficients")
print("• Efficiency Ratios: F1 Score per unit of computational complexity")
print("• Ensemble FLOPs: Summed across all sub-models (represents total compute)")
print("="*80)

print(f"\n✅ Analysis Complete!")


MODEL COMPLEXITY & COMPUTATIONAL ANALYSIS (+ FLOPs)

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
TABLE 1: INDIVIDUAL MODELS - COMPLEXITY METRICS (Including FLOPs Estimation)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

                   Model     Type  Input Dims Parameters FLOPs/Sample
           Model 1 (MLP)      MLP        1668    468,546      936,192
Model 1.1 (MLP+Logistic) Ensemble        1668    470,215      939,528
Model 1.2 (MLP+LightGBM) Ensemble        1668  2,136,546      937,592
    Model 1.3 (Stacking) Stacking        1668  2,136,549      937,596
           Model 2 (MLP)      MLP        1724    229,186      457,984
Model 2.1 (MLP+Logistic) Ensemble        1724    230,911      461,432
Model 2.2 (MLP+LightGBM) Ensemble        1724  1,953,186      459,384
    Model 2.3 (Stacking) Stacking        1724  1,953,189      459,